# 🔐 UPI Payment Fraud Detection
### End-to-End Machine Learning Project

---

**Objective:** Detect fraudulent UPI transactions using classical machine learning.

**Dataset:** `upi_transactions.csv` — synthetic UPI payment records with 17 features.

**Pipeline:**
1. Data loading & cleaning
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. Model training (Logistic Regression, Random Forest, Gradient Boosting, XGBoost)
5. Model evaluation (ROC-AUC, Precision-Recall, Confusion Matrix)
6. Live prediction demo

---

## 📦 1. Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy scikit-learn imbalanced-learn xgboost plotly matplotlib seaborn streamlit

In [ ]:
import os, warnings, pickle
import numpy  as np
import pandas as pd
import matplotlib.pyplot   as plt
import matplotlib.ticker   as mticker
import seaborn             as sns
import plotly.express      as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline         import Pipeline
from sklearn.metrics          import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score,
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print('XGBoost not installed — skipping')
    HAS_XGB = False

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    print('imbalanced-learn not installed — SMOTE disabled')
    HAS_SMOTE = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
print('All imports successful ✅')

## 📂 2. Load & Inspect Data

In [ ]:
DATA_PATH = 'data/upi_transactions.csv'

df = pd.read_csv(DATA_PATH)

# Standardise column names
df.columns = [
    c.strip().lower()
     .replace(' ', '_')
     .replace('(inr)', 'inr')
     .replace('_(inr)', '_inr')
    for c in df.columns
]
if 'amount_(inr)' in df.columns:
    df.rename(columns={'amount_(inr)': 'amount_inr'}, inplace=True)

print(f'Shape       : {df.shape}')
print(f'Columns     : {df.columns.tolist()}')
df.head(5)

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
df.describe().T

In [ ]:
fraud_count = df['fraud_flag'].value_counts()
print('Fraud distribution:')
print(fraud_count)
print(f'\nFraud rate: {fraud_count[1]/len(df)*100:.2f}%')

## 🧹 3. Data Cleaning

In [ ]:
# Parse timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Numeric casts
df['amount_inr']  = pd.to_numeric(df['amount_inr'],  errors='coerce')
df['fraud_flag']  = pd.to_numeric(df['fraud_flag'],  errors='coerce').astype(int)
df['hour_of_day'] = pd.to_numeric(df['hour_of_day'], errors='coerce')
df['is_weekend']  = pd.to_numeric(df['is_weekend'],  errors='coerce')

# Drop rows with nulls in critical columns
df.dropna(subset=['timestamp', 'amount_inr'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Clean shape: {df.shape}')
print('Remaining nulls:', df.isnull().sum().sum())

## 📊 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 Class distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

counts = df['fraud_flag'].value_counts()
ax[0].bar(['Genuine', 'Fraud'], counts.values, color=['#3b82d4', '#ef4444'])
ax[0].set_title('Transaction Count by Class')
ax[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

ax[1].pie(counts.values, labels=['Genuine', 'Fraud'],
          colors=['#3b82d4', '#ef4444'], autopct='%1.1f%%',
          startangle=140, wedgeprops=dict(width=0.5))
ax[1].set_title('Class Distribution (Donut)')

plt.suptitle('UPI Fraud Detection — Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

genuine = df[df['fraud_flag'] == 0]['amount_inr']
fraud   = df[df['fraud_flag'] == 1]['amount_inr']

axes[0].hist(genuine, bins=50, color='#3b82d4', alpha=0.7, label='Genuine')
axes[0].hist(fraud,   bins=50, color='#ef4444', alpha=0.7, label='Fraud')
axes[0].set_title('Amount Distribution (INR)')
axes[0].set_xlabel('Amount (INR)')
axes[0].legend()

axes[1].boxplot([genuine, fraud], labels=['Genuine', 'Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='#dbeafe'),
                medianprops=dict(color='#1e40af', linewidth=2))
axes[1].set_title('Amount Box Plot by Class')
axes[1].set_ylabel('Amount (INR)')

plt.tight_layout()
plt.show()

print(f'Avg Genuine Amount : ₹{genuine.mean():.0f}')
print(f'Avg Fraud Amount   : ₹{fraud.mean():.0f}')

In [ ]:
# 4.3 Fraud by hour of day
hourly = df.groupby(['hour_of_day', 'fraud_flag']).size().unstack(fill_value=0)
hourly.columns = ['Genuine', 'Fraud']

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(24)
w = 0.4
ax.bar(x - w/2, hourly['Genuine'], w, label='Genuine', color='#3b82d4')
ax.bar(x + w/2, hourly['Fraud'],   w, label='Fraud',   color='#ef4444')
ax.set_xticks(x)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Transaction Count')
ax.set_title('Transactions by Hour of Day')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Fraud rate by merchant category
cat_fraud = (
    df.groupby('merchant_category')['fraud_flag']
    .mean()
    .sort_values(ascending=False) * 100
)

fig, ax = plt.subplots(figsize=(10, 5))
cat_fraud.plot(kind='bar', ax=ax, color='#ef4444', edgecolor='white')
ax.set_title('Fraud Rate (%) by Merchant Category')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Fraud rate by state
state_fraud = (
    df.groupby('sender_state')['fraud_flag']
    .mean()
    .sort_values(ascending=False) * 100
)

fig, ax = plt.subplots(figsize=(12, 5))
state_fraud.plot(kind='bar', ax=ax, color='#f59e0b', edgecolor='white')
ax.set_title('Fraud Rate (%) by Sender State')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('State')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 4.6 Fraud by network and device type
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

net_fraud = df.groupby('network_type')['fraud_flag'].mean().sort_values(ascending=False) * 100
net_fraud.plot(kind='bar', ax=axes[0], color='#8b5cf6', edgecolor='white')
axes[0].set_title('Fraud Rate by Network Type')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].tick_params(axis='x', rotation=0)

dev_fraud = df.groupby('device_type')['fraud_flag'].mean().sort_values(ascending=False) * 100
dev_fraud.plot(kind='bar', ax=axes[1], color='#10b981', edgecolor='white')
axes[1].set_title('Fraud Rate by Device Type')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 4.7 Correlation heatmap
num_cols = ['amount_inr', 'hour_of_day', 'is_weekend', 'fraud_flag']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 🔧 5. Feature Engineering

In [ ]:
AGE_ORDER = ['18-25', '26-35', '36-45', '46-55', '56+']
AGE_MAP   = {a: i for i, a in enumerate(AGE_ORDER)}

df_feat = df.copy()

# ── Time features ─────────────────────────────────────────────────────────
df_feat['month']      = df_feat['timestamp'].dt.month
df_feat['quarter']    = df_feat['timestamp'].dt.quarter
df_feat['is_night']   = ((df_feat['hour_of_day'] >= 22) | (df_feat['hour_of_day'] <= 5)).astype(int)
df_feat['is_morning'] = ((df_feat['hour_of_day'] >= 6)  & (df_feat['hour_of_day'] <= 10)).astype(int)

# ── Amount ────────────────────────────────────────────────────────────────
df_feat['log_amount'] = np.log1p(df_feat['amount_inr'])

# ── Same bank ─────────────────────────────────────────────────────────────
df_feat['same_bank']  = (df_feat['sender_bank'] == df_feat['receiver_bank']).astype(int)

# ── Age ordinal ───────────────────────────────────────────────────────────
df_feat['sender_age_enc']   = df_feat['sender_age_group'].map(AGE_MAP).fillna(-1).astype(int)
df_feat['receiver_age_enc'] = df_feat['receiver_age_group'].map(AGE_MAP).fillna(-1).astype(int)
df_feat['age_diff']         = (df_feat['sender_age_enc'] - df_feat['receiver_age_enc']).abs()

# ── One-hot encode ────────────────────────────────────────────────────────
OHE_COLS = [
    'transaction_type', 'merchant_category', 'transaction_status',
    'sender_state', 'sender_bank', 'receiver_bank',
    'device_type', 'network_type', 'day_of_week',
]
df_feat = pd.get_dummies(df_feat, columns=OHE_COLS, drop_first=False, dtype=int)

# ── Drop non-model columns ────────────────────────────────────────────────
DROP = ['transaction_id', 'timestamp', 'sender_age_group', 'receiver_age_group']
df_feat.drop(columns=[c for c in DROP if c in df_feat.columns], inplace=True)

print(f'Feature matrix shape: {df_feat.shape}')
print(f'Features: {df_feat.columns.tolist()[:15]} ...')

In [ ]:
y = df_feat['fraud_flag']
X = df_feat.drop(columns=['fraud_flag'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Train size : {X_train.shape[0]:,}  | Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Test size  : {X_test.shape[0]:,}   | Fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

In [ ]:
# SMOTE oversampling
if HAS_SMOTE:
    sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f'After SMOTE — Train size: {len(X_train_res):,}  | Fraud: {y_train_res.sum():,}')
else:
    X_train_res, y_train_res = X_train, y_train
    print('SMOTE not applied')

## 🤖 6. Model Training

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000,
                                   random_state=RANDOM_STATE, solver='lbfgs')),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        subsample=0.8, random_state=RANDOM_STATE
    ),
}

if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=10,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0
    )

print(f'Models to train: {list(models.keys())}')

In [ ]:
results = {}

for name, clf in models.items():
    print(f'Training [{name}] ...', end=' ')
    clf.fit(X_train_res, y_train_res)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    roc_auc  = roc_auc_score(y_test, y_prob)
    avg_prec = average_precision_score(y_test, y_prob)
    report   = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {
        'model':    clf,
        'y_pred':   y_pred,
        'y_prob':   y_prob,
        'roc_auc':  round(roc_auc, 4),
        'avg_prec': round(avg_prec, 4),
        'f1_fraud': round(report.get('1', {}).get('f1-score', 0), 4),
        'recall':   round(report.get('1', {}).get('recall', 0), 4),
        'precision_fraud': round(report.get('1', {}).get('precision', 0), 4),
        'accuracy': round(report.get('accuracy', 0), 4),
        'report':   classification_report(y_test, y_pred),
    }
    print(f'ROC-AUC={roc_auc:.4f}  F1(fraud)={results[name]["f1_fraud"]:.4f} ✅')

print('\nAll models trained!')

## 📈 7. Model Evaluation

In [ ]:
# Summary comparison table
summary = pd.DataFrame([
    {
        'Model':          name,
        'ROC-AUC':        res['roc_auc'],
        'Avg Precision':  res['avg_prec'],
        'F1 (Fraud)':     res['f1_fraud'],
        'Recall (Fraud)': res['recall'],
        'Accuracy':       res['accuracy'],
    }
    for name, res in results.items()
]).set_index('Model')

summary.style.highlight_max(axis=0, color='#bbf7d0')

In [ ]:
# ROC Curves
colors = ['#3b82d4', '#ef4444', '#10b981', '#f59e0b']

fig, ax = plt.subplots(figsize=(9, 7))
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    roc_val     = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={roc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curves
fig, ax = plt.subplots(figsize=(9, 7))
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ap           = average_precision_score(y_test, res['y_prob'])
    ax.plot(rec, prec, lw=2, color=color, label=f'{name} (AP={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — All Models')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Genuine','Fraud'],
                yticklabels=['Genuine','Fraud'],
                ax=ax)
    ax.set_title(f'{name}\nROC-AUC={res["roc_auc"]}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — best tree model
best_name = max(results, key=lambda n: results[n]['roc_auc'])
best_clf  = results[best_name]['model']

estimator = best_clf
if hasattr(best_clf, 'named_steps'):   # unwrap Pipeline
    estimator = best_clf.named_steps.get('clf', best_clf)

if hasattr(estimator, 'feature_importances_'):
    fi = pd.DataFrame({'feature': X.columns, 'importance': estimator.feature_importances_})
    fi = fi.sort_values('importance', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(fi['feature'][::-1], fi['importance'][::-1], color='#3b82d4')
    ax.set_title(f'Top 20 Feature Importances — {best_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances not available for this model type.')

In [ ]:
# Full classification reports
for name, res in results.items():
    print(f'{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(res['report'])

## 🔍 8. Live Prediction Demo

In [ ]:
# Example transaction — change values to test
sample = {
    'timestamp':          pd.Timestamp('2024-06-15 02:15:00'),  # 2 AM — night
    'transaction_type':   'P2P',
    'merchant_category':  'Shopping',
    'amount_inr':         49500.0,                               # large amount
    'transaction_status': 'SUCCESS',
    'sender_age_group':   '18-25',
    'receiver_age_group': '56+',
    'sender_state':       'Delhi',
    'sender_bank':        'ICICI',
    'receiver_bank':      'SBI',
    'device_type':        'Web',
    'network_type':       '3G',
    'fraud_flag':         0,                                     # placeholder
    'hour_of_day':        2,
    'day_of_week':        'Sunday',
    'is_weekend':         1,
}

# Append to full df and encode
full = pd.concat([df, pd.DataFrame([sample])], ignore_index=True)

AGE_MAP_INV = {a: i for i, a in enumerate(AGE_ORDER)}
OHE_COLS_PRED = [
    'transaction_type', 'merchant_category', 'transaction_status',
    'sender_state', 'sender_bank', 'receiver_bank',
    'device_type', 'network_type', 'day_of_week',
]

full['month']      = full['timestamp'].dt.month
full['quarter']    = full['timestamp'].dt.quarter
full['is_night']   = ((full['hour_of_day'] >= 22) | (full['hour_of_day'] <= 5)).astype(int)
full['is_morning'] = ((full['hour_of_day'] >= 6)  & (full['hour_of_day'] <= 10)).astype(int)
full['log_amount'] = np.log1p(full['amount_inr'])
full['same_bank']  = (full['sender_bank'] == full['receiver_bank']).astype(int)
full['sender_age_enc']   = full['sender_age_group'].map(AGE_MAP_INV).fillna(-1).astype(int)
full['receiver_age_enc'] = full['receiver_age_group'].map(AGE_MAP_INV).fillna(-1).astype(int)
full['age_diff']         = (full['sender_age_enc'] - full['receiver_age_enc']).abs()

full_enc = pd.get_dummies(full, columns=OHE_COLS_PRED, drop_first=False, dtype=int)
full_enc.drop(columns=['transaction_id','timestamp','sender_age_group','receiver_age_group'],
              inplace=True, errors='ignore')

input_row = full_enc.iloc[[-1]].drop(columns=['fraud_flag'], errors='ignore')
input_aligned = input_row.reindex(columns=X.columns, fill_value=0)

print('━' * 40)
print('  FRAUD PREDICTION RESULTS')
print('━' * 40)
for name, res in results.items():
    prob  = res['model'].predict_proba(input_aligned)[0, 1]
    label = '🚨 FRAUD' if prob >= 0.5 else '✅ GENUINE'
    print(f'  {name:<25}  {prob*100:5.1f}%  {label}')
print('━' * 40)

## 💾 9. Save Models

In [ ]:
os.makedirs('models', exist_ok=True)

for name, res in results.items():
    path = f'models/{name.replace(" ", "_").lower()}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(res['model'], f)
    print(f'Saved: {path}')

# Save feature column list for inference
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)
print('Saved: models/feature_columns.pkl')

## ✅ 10. Conclusion

### Key Findings

| Insight | Detail |
|---|---|
| **Class imbalance** | ~10% fraud rate — handled via SMOTE + class_weight |
| **Best model** | XGBoost / Gradient Boosting achieve highest ROC-AUC |
| **Top fraud signals** | Amount, hour of day (late night), same_bank flag, age_diff |
| **Risky categories** | Fuel, Healthcare — highest fraud rates |
| **Risky networks** | 3G shows elevated fraud rate vs 5G/4G |

### Next Steps
- Hyperparameter tuning with `GridSearchCV` / `Optuna`
- Add behavioural features (velocity, rolling averages per sender)
- Deploy as REST API with FastAPI
- Real-time monitoring dashboard with Streamlit

---

**Launch the interactive dashboard:**
```bash
streamlit run dashboard.py
```